# Phase 1 Analysis Notebook (Draft)
Lightweight inspection notebook for `data/10_mention_detection` outputs.

This is intentionally limited before Phase 3 disambiguation and deduplication decisions are finalized.

## 1) Project Setup
Resolve repository paths and load phase contracts.

In [ ]:
from pathlib import Path
import sys
import re

import pandas as pd


def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / "data").exists() and (cur / "speakermining" / "src").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError("Repository root not found.")


ROOT = find_repo_root(Path.cwd())
SRC = ROOT / "speakermining" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from process.mention_detection.config import (
    EPISODE_COLUMNS,
    FILE_EPISODES,
    FILE_PERSON_MENTIONS,
    FILE_PUBLIKATION,
    FILE_SEASONS,
    FILE_TOPIC_MENTIONS,
    PERSON_MENTION_COLUMNS,
    PHASE_DIR,
    PUBLIKATION_COLUMNS,
    SEASON_COLUMNS,
    TOPIC_MENTION_COLUMNS,
)

phase_dir = ROOT / PHASE_DIR
phase_dir

NameError: name 'bootstrap_notebook_paths' is not defined

## 2) Load Phase 1 Tables
Missing files are represented as empty dataframes with contract columns.

In [ ]:
def load_or_empty(path: Path, columns: list[str]) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=columns)
    df = pd.read_csv(path)
    for column in columns:
        if column not in df.columns:
            df[column] = pd.NA
    return df

episodes_df = load_or_empty(phase_dir / FILE_EPISODES, EPISODE_COLUMNS)
publications_df = load_or_empty(phase_dir / FILE_PUBLIKATION, PUBLIKATION_COLUMNS)
seasons_df = load_or_empty(phase_dir / FILE_SEASONS, SEASON_COLUMNS)
persons_df = load_or_empty(phase_dir / FILE_PERSON_MENTIONS, PERSON_MENTION_COLUMNS)
topics_df = load_or_empty(phase_dir / FILE_TOPIC_MENTIONS, TOPIC_MENTION_COLUMNS)

{
    'episodes_rows': len(episodes_df),
    'publications_rows': len(publications_df),
    'seasons_rows': len(seasons_df),
    'persons_rows': len(persons_df),
    'topics_rows': len(topics_df),
}

## 3) Lightweight Coverage Overview

In [ ]:
overview = pd.DataFrame([
    {
        'table': 'episodes',
        'rows': len(episodes_df),
        'unique_ids': episodes_df['episode_id'].nunique() if 'episode_id' in episodes_df.columns else 0,
    },
    {
        'table': 'publications',
        'rows': len(publications_df),
        'unique_ids': publications_df['publikation_id'].nunique() if 'publikation_id' in publications_df.columns else 0,
    },
    {
        'table': 'seasons',
        'rows': len(seasons_df),
        'unique_ids': seasons_df['season_id'].nunique() if 'season_id' in seasons_df.columns else 0,
    },
    {
        'table': 'persons',
        'rows': len(persons_df),
        'unique_ids': persons_df['mention_id'].nunique() if 'mention_id' in persons_df.columns else 0,
    },
    {
        'table': 'topics',
        'rows': len(topics_df),
        'unique_ids': topics_df['mention_id'].nunique() if 'mention_id' in topics_df.columns else 0,
    },
])

overview

## 4) Mention Quality Snapshot
Confidence-aware checks without assuming final entity resolution quality.

In [ ]:
persons_conf = pd.to_numeric(persons_df['confidence'], errors='coerce') if 'confidence' in persons_df.columns else pd.Series(dtype='float64')
topics_conf = pd.to_numeric(topics_df['confidence'], errors='coerce') if 'confidence' in topics_df.columns else pd.Series(dtype='float64')

quality = {
    'persons_avg_confidence': float(persons_conf.mean()) if not persons_conf.empty else None,
    'persons_invalid_confidence_rows': int(persons_conf.isna().sum()) if not persons_conf.empty else 0,
    'persons_missing_beschreibung': int(persons_df['beschreibung'].isna().sum()) if 'beschreibung' in persons_df.columns else 0,
    'topics_avg_confidence': float(topics_conf.mean()) if not topics_conf.empty else None,
    'topics_invalid_confidence_rows': int(topics_conf.isna().sum()) if not topics_conf.empty else 0,
}

person_rules = persons_df['parsing_rule'].value_counts().rename_axis('parsing_rule').to_frame('rows') if 'parsing_rule' in persons_df.columns else pd.DataFrame()
topic_rules = topics_df['parsing_rule'].value_counts().rename_axis('parsing_rule').to_frame('rows') if 'parsing_rule' in topics_df.columns else pd.DataFrame()

quality, person_rules.head(10), topic_rules.head(10)

## 5) Early Legacy-Inspired Sanity Checks (Draft)
These checks are only weak signals and should not drive final conclusions before later phases.

In [ ]:
episodes_without_person_mentions = int(
    episodes_df['episode_id'].nunique() - persons_df['episode_id'].nunique()
) if ('episode_id' in episodes_df.columns and 'episode_id' in persons_df.columns) else None

names_ending_with_von = int(
    persons_df['name'].fillna('').str.upper().str.endswith(' VON').sum()
) if 'name' in persons_df.columns else None

thevessen_variant_count = int(
    episodes_df['infos'].fillna('').str.contains(r'THEVE(?:ß|SS)EN', regex=True, case=False).sum()
) if 'infos' in episodes_df.columns else None

{
    'episodes_without_person_mentions': episodes_without_person_mentions,
    'names_ending_with_von': names_ending_with_von,
    'infos_thevessen_variant_count': thevessen_variant_count,
}